In [1]:
import numpy as np
import pandas as pd
from joblib import load

bundle = load("models/dimmad_medmed_relaiss_bundle.joblib")

feature_cols = bundle["feature_cols"]
imputer = bundle["imputer"]
scaler = bundle["scaler"]
model = bundle["model"]

FileNotFoundError: [Errno 2] No such file or directory: 'models/dimmad_medmed_relaiss_bundle.joblib'

In [10]:
import os

print("Current directory:", os.getcwd())
print("Files here:", os.listdir())
os.chdir("/Users/jennakempster-taylor/re-laiss")
print("Files here:", os.listdir())

Current directory: /Users/jennakempster-taylor/re-laiss/outputs
Files here: ['Untitled1.ipynb', 'Untitled.ipynb', 'ranked_dimmadscores.csv', 'dimmad_ranked_medmed.csv', '.ipynb_checkpoints', 'relaissranked_medmed_.csv', 'dimmad_anomaly_scores.csv']
Files here: ['lightcurves', 'ztfdata.txt', '.DS_Store', 'reference_20k.csv', 'LICENSE', 'Untitled.ipynb', 'iforest_artifacts.joblib', 'experiments', 'dimmad_anomaly_rankings.csv', 'pyproject.toml', 'dimmad_medmed_bundle.joblib', 'tests', 'alerce_200_recent_iforest.csv', 'models', 'README.md', 'reference_matched_with_alerce.csv', 'DimmadTest.ipynb', '.gitignore', 'static', 'IsoForestDur.ipynb', 'examples', 'reference_unmatched_for_dimmad.csv', '.github', 'reference_20kk.csv', 'irsa_cutouts', '.ipynb_checkpoints', 'DimmadTest_cleaned.ipynb', '.git', 'data', 'outputs', 'notebooks', '.zenodo.json', 'robust_anomaly_cutouts', 'DIMMAD_scores_medmed_minmed.csv', 'src']


In [12]:
bundle = load("models/dimmad_medmed_relaiss_bundle.joblib")

feature_cols = bundle["feature_cols"]
imputer = bundle["imputer"]
scaler = bundle["scaler"]
model = bundle["model"]

In [16]:
from joblib import load
import numpy as np
import pandas as pd

# ── Load extracted features ───────────────────────────────────────────────────
df_all = pd.read_csv("lsst_extracted_features.csv")
print(f"Loaded {len(df_all)} objects")

# ── Load model artifacts ──────────────────────────────────────────────────────
iso_art    = load("iforest_artifacts.joblib")
dimmad_art = load("models/dimmad_medmed_relaiss_bundle.joblib")

Loaded 221 objects


In [18]:
iso              = iso_art["iso"]
knn_imp          = iso_art["knn_imp"]
iso_cols         = iso_art["numeric_feature_cols"]

dimmad_model     = dimmad_art["model"]
dimmad_imputer   = dimmad_art["imputer"]
dimmad_scaler    = dimmad_art["scaler"]
dimmad_cols      = dimmad_art["feature_cols"]

In [20]:
# ── Prepare feature matrix ────────────────────────────────────────────────────
# Fill any missing columns with NaN
for c in iso_cols:
    if c not in df_all.columns:
        df_all[c] = np.nan

In [22]:
# ── Isolation Forest scoring ──────────────────────────────────────────────────
X_iso     = df_all[iso_cols].replace([np.inf, -np.inf], np.nan)
X_iso_imp = knn_imp.transform(X_iso)

iso_scores  = iso.decision_function(X_iso_imp)
iso_pred    = iso.predict(X_iso_imp)
iso_anomaly = (iso_pred == -1).astype(int)

In [24]:
# ── DiMMAD scoring ────────────────────────────────────────────────────────────
for c in dimmad_cols:
    if c not in df_all.columns:
        df_all[c] = np.nan

X_dim        = df_all[dimmad_cols].replace([np.inf, -np.inf], np.nan)
X_dim_imp    = dimmad_imputer.transform(X_dim)
X_dim_scaled = dimmad_scaler.transform(X_dim_imp)
dimmad_scores = dimmad_model.score_samples(X_dim_scaled)

In [26]:
# ── Combine results ───────────────────────────────────────────────────────────
results = pd.DataFrame({
    "lsst_id":       df_all["lsst_dia_object_id"],
    "iso_score":     iso_scores,
    "iso_anomaly":   iso_anomaly,
    "iso_rank":      pd.Series(iso_scores).rank(method="first", ascending=True).astype(int),
    "dimmad_score":  dimmad_scores,
    "dimmad_rank":   pd.Series(dimmad_scores).rank(method="first", ascending=False).astype(int),
})

results.to_csv("lsst_anomaly_scores.csv", index=False)
print(f"\nIsoForest anomaly rate: {iso_anomaly.mean():.3f}  ({iso_anomaly.sum()} flagged)")
print(f"\nTop 10 anomalies by Isolation Forest:")
display(results.sort_values("iso_rank").head(10))
print(f"\nTop 10 anomalies by DiMMAD:")
display(results.sort_values("dimmad_rank").head(10))


IsoForest anomaly rate: 1.000  (221 flagged)

Top 10 anomalies by Isolation Forest:


,lsst_id,iso_score,iso_anomaly,iso_rank,dimmad_score,dimmad_rank
186,313761042386124904,-0.201325,1,1,-0.256995,189
161,313897383783038984,-0.186324,1,2,-0.221555,151
31,313963359375458336,-0.184281,1,3,-0.248366,174
138,170028526789460018,-0.182080,1,4,-0.217894,148
41,170028485783322694,-0.181429,1,5,-0.232494,164
3,313853517475348486,-0.180123,1,6,-0.120741,37
140,170028527011758122,-0.178337,1,7,-0.231662,162
210,313853517574963301,-0.177475,1,8,-0.128107,46
195,313871013678940198,-0.177415,1,9,-0.126381,43
83,170032906681450641,-0.177347,1,10,-0.226902,159



Top 10 anomalies by DiMMAD:


,lsst_id,iso_score,iso_anomaly,iso_rank,dimmad_score,dimmad_rank
85,313871013252694140,-0.142601,1,126,-0.029965,1
176,313893023086280813,-0.012353,1,221,-0.043309,2
215,313853517424492644,-0.149575,1,102,-0.073391,3
62,313637935744286779,-0.135661,1,135,-0.076860,4
190,313699506405244961,-0.150724,1,96,-0.078530,5
189,313756673012400719,-0.138749,1,132,-0.080999,6
35,313690717366517778,-0.143316,1,123,-0.082601,7
50,313941449021325389,-0.074881,1,203,-0.103207,8
212,313853517448085571,-0.149813,1,99,-0.108090,9
79,313871013398970376,-0.166066,1,19,-0.108303,10


In [ ]:
print(f"Min score: {iso_scores.min():.4f}")
print(f"Max score: {iso_scores.max():.4f}")
print(f"Mean score: {iso_scores.mean():.4f}")
print(f"Threshold: {tau}")  # your -0.215